# 21 cm line-emission simulation with SKA-SDP

This compact example creates a synthetic line source, simulates one frequency channel with the SDP backend, and produces dirty, PSF, and restored images through Karabo's backend-neutral imaging interface.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord

from karabo.imaging.imager_base import DirtyImagerConfig
from karabo.imaging.imager_factory import ImagingBackend
from karabo.simulation.interferometer import InterferometerSimulation
from karabo.simulation.line_emission import CircleSkyRegion, line_emission_pipeline
from karabo.simulation.line_emission_helpers import convert_frequency_to_z
from karabo.simulation.observation import Observation
from karabo.simulation.sky_model import SkyModel
from karabo.simulation.telescope import Telescope
from karabo.simulator_backend import SimulatorBackend
from karabo.util.file_handler import FileHandler

%matplotlib inline

The deliberately small fixture keeps the notebook runnable on a laptop. Production studies can increase the number of channels, time steps, pointings, and image pixels without changing the backend-selection pattern.

In [ ]:
ra_deg, dec_deg = 20.0, -30.0
start_frequency_hz = 1.0e9
channel_width_hz = 1.0e6
source_redshift = float(
    convert_frequency_to_z(start_frequency_hz + channel_width_hz / 2)
)

source = np.zeros((1, SkyModel.SOURCES_COLS), dtype=float)
source[0, [0, 1, 2, 6, 12, 13]] = [
    ra_deg,
    dec_deg,
    1.0,
    start_frequency_hz,
    source_redshift,
    source_redshift,
]
sky = SkyModel(source)

pointings = [
    CircleSkyRegion(
        center=SkyCoord(ra_deg, dec_deg, unit="deg", frame="icrs"),
        radius=2 * u.deg,
    )
]
observation = Observation(
    phase_centre_ra_deg=ra_deg,
    phase_centre_dec_deg=dec_deg,
    start_date_and_time=datetime(2000, 3, 20, 12, 6, 39),
    length=timedelta(seconds=1),
    number_of_time_steps=1,
    start_frequency_hz=start_frequency_hz,
    frequency_increment_hz=channel_width_hz,
    number_of_channels=1,
)

In [ ]:
simulator_backend = SimulatorBackend.SDP
imaging_backend = ImagingBackend.SDP
telescope = Telescope.constructor("ASKAP", backend=simulator_backend)
interferometer = InterferometerSimulation(
    channel_bandwidth_hz=channel_width_hz,
    time_average_sec=1.0,
    ignore_w_components=True,
    use_gpus=False,
    use_dask=False,
)
dirty_config = DirtyImagerConfig(
    imaging_npixel=128,
    imaging_cellsize=np.radians(4.0) / 128,
)
output_directory = Path(
    FileHandler().get_tmp_dir(
        prefix="line-emission-sdp-",
        purpose="SDP line-emission notebook",
    )
)

visibilities, dirty_images, psf_images, restored_images = line_emission_pipeline(
    output_base_directory=output_directory,
    pointings=pointings,
    sky_model=sky,
    observation_details=observation,
    telescope=telescope,
    interferometer=interferometer,
    simulator_backend=simulator_backend,
    dirty_imager_config=dirty_config,
    imaging_backend=imaging_backend,
)

In [ ]:
dirty = dirty_images[0][0]
psf = psf_images[0][0]
restored = restored_images[0][0]

assert visibilities[0][0].format == "MS"
assert dirty.data.shape == psf.data.shape == restored.data.shape
assert np.isfinite(restored.data).all()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, image, title in zip(
    axes,
    (dirty, psf, restored),
    ("Dirty", "PSF", "Restored"),
):
    plotted = axis.imshow(image.get_squeezed_data(), origin="lower")
    axis.set_title(title)
    fig.colorbar(plotted, ax=axis, shrink=0.75)
fig.tight_layout()